# Create summaries -- Sept 2025 version

In [ ]:
from datetime import datetime, timedelta
import geopandas as gpd
import numpy as np
import pandas as pd
from pathlib import Path

from gtamodel_tools.config import Config
from gtamodel_tools.matrix.matrix import Matrix
from gtamodel_tools.network import Network
from gtamodel_tools.microsim.microsim import MicroSim
import gtamodel_tools.common.spatial_aggregator as sa
from gtamodel_tools.common.plotting import plot_line_profiles, plot_tlfds
# from gtamodel_tools.matrix.read_emme_matrix import read_mdf


# Traffic summary tools
from gtamodel_tools.summaries.traffic_summaries import summarize_traffic_vkt
from gtamodel_tools.summaries.traffic_summaries import summarize_traffic_vht

# Transit summary tools
from gtamodel_tools.summaries.transit_summaries import summarize_transit_boardings_by_operator
from gtamodel_tools.summaries.transit_summaries import summarize_transit_pkt_by_operator
from gtamodel_tools.summaries.transit_summaries import summarize_transit_line_profiles
from gtamodel_tools.summaries.transit_summaries import summarize_at_transit_countposts

import gtamodel_tools.enums.microsim as ems


# Inputs

In [ ]:
model_root_dir = Path(r"Complete here")

summary_support_dir = Path(r'Complete here')
config_fp = summary_support_dir / 'city_of_toronto_2025TTCsummaries_Networks1-3.yml'   # For N1, N2, N3
# config_fp = summary_support_dir / 'city_of_toronto_2025TTCsummaries_Network0.yml'   # for N0



zonefiles_dir = Path(r'Complete here')
pds_fp = r"Complete here"

# Start the timer

In [ ]:
run_start_time = datetime.now()
run_start_time

# Setup

### Read config file

In [ ]:
config = Config(model_root_dir, config_fp)

In [ ]:
screenlines = gpd.read_file(config.screenlines_fp)
screenlines = screenlines.set_index('Name2')

### Read in Centres shapefile

In [ ]:
centres_gdf = gpd.read_file(summary_support_dir / 'centres.gpkg')
centres_gdf = centres_gdf.set_index('NAME')

### Read planning districts shapefile

In [ ]:
pds_gdf = gpd.read_file(pds_fp)
pds_gdf = pds_gdf.set_index('PD_no')
pds_gdf = pds_gdf[['geometry']]
pds_gdf = pds_gdf.to_crs('epsg:2952')

### Read in GTAModel zone pd and regions files

In [ ]:
zone_pds = pd.read_csv(
    zonefiles_dir / 'Zones.csv', 
    index_col='Zone#',
    usecols=['Zone#', 'PD']
).squeeze()
zone_pds[0] = 0

zone_regions = pd.read_csv(
    zonefiles_dir / 'Regions.csv', 
    index_col='Zone',
    usecols=['Zone', 'Region']
).squeeze()
zone_regions[0] = 0

### Read in a network to prepare spatial aggregations

In [ ]:
# Do a spatial join to find the zones that are in the centres
# First read in any base network
net = Network(config)
net.read_from_nwp(config.networks_subdir / f'AM.nwp')

# Define PD, region and centres spatial aggregations

In [ ]:
# For zones, uses the zones and regions input files. We've found this 
# approach to be useful as previous testing has found errors in these files.
sa_zns_pd = sa.create_spatial_aggregator(
    aggregation_type='one_level_mapping', name='sa_zns_pd', lvl1_mapping=zone_pds)
sa_zns_reg = sa.create_spatial_aggregator(
    aggregation_type='one_level_mapping', name='sa_zns_reg', lvl1_mapping=zone_regions)

# For nodes, we'll do a spatial aggregation to the TTS
# Planning Districts shapefile
sa_nds_pd = sa.create_spatial_aggregator(
    aggregation_type='shapefile', name='sa_zns_pd', points=net.nodes, areas=pds_gdf)

# For centres, we'll do spatial aggregation to centres shapefile
centroids = net.nodes.loc[net.nodes['is_centroid']].copy()
sz_zns_cntrs = sa.create_spatial_aggregator(
    aggregation_type='shapefile', name='sa_zns_pd', points=centroids, areas=centres_gdf)

# Create directories to hold summaries

In [ ]:
summaries_dir = model_root_dir / 'Summaries'
summaries_dir.mkdir(exist_ok=True)

transit_dir = summaries_dir / 'Transit'
transit_dir.mkdir(exist_ok=True)

auto_dir = summaries_dir / 'Auto'
auto_dir.mkdir(exist_ok=True)

ms_dir = summaries_dir / 'Demand'
ms_dir.mkdir(exist_ok=True)

# Transit Network Summaries

Remember, transit assignments are period flows, hence summing to daily values
is simply summing the period-level results

### Read in transit networks

Store in a dictionary to access later

In [ ]:
tnets = {}
dly_tnets = []
for tp in config.time_periods.keys():
    if config.time_periods[tp]['is_assigned']:
        tnets[tp] = Network(
            config, 
            config.time_periods[tp]['start_time'],
            config.time_periods[tp]['end_time'],
            None,
            config.time_periods[tp]['transit_phf']
        )
        tnets[tp].read_from_nwp(config.networks_subdir / f'{tp}_Transit.nwp')
        tnets[tp] = tnets[tp].collapse_hypernetwork()
        dly_tnets.append(tnets[tp])

### Summarize transit boardings by operator

In [ ]:
for tp, net in tnets.items():
    summarize_transit_boardings_by_operator(net).to_csv(
        transit_dir / f'{tp}_boardings_by_operator.csv')
summarize_transit_boardings_by_operator(dly_tnets).to_csv(
    transit_dir / f'daily_boardings_by_operator.csv')

### Calculate line profiles

In [ ]:
for tp, net in tnets.items():
    print(f'Processing time period: {tp}:')
    line_profiles = summarize_transit_line_profiles(net)
    line_profiles.to_csv(
        transit_dir / f'{tp}_line_profiles.csv')
    plot_line_profiles(
        line_profiles, 
        fp=transit_dir / f'{tp}_line_profiles.png',
        figsize=(8, 8), 
        fontsize=14, 
    )

### Export volumes at transit countposts

In [ ]:
for tp, net in tnets.items():
    summarize_at_transit_countposts(net).to_csv(
        transit_dir / f'{tp}_countposts.csv')
# Calculate daily values at countposts and recalculate VCR
dly_transit_countposts = summarize_at_transit_countposts(dly_tnets)
dly_transit_countposts['vcr'] = \
dly_transit_countposts['volume'] / dly_transit_countposts['capacity']
dly_transit_countposts.to_csv(
    transit_dir / f'daily_countposts.csv')

### Linked trips using TTC

In [ ]:
# A custom tool was added to the City of Toronto's GTAModel XTMF config that 
# runs a path-based transit analysis to calculate the demand using the TTC.
# This is to calculate what the TTC calls 'Linked Trips' using their system.

uses_ttc = {}
dly_uses_ttc = 0
for tp in config.time_periods:
    if config.time_periods[tp]['is_assigned']: 
        print(f'Processing time period: {tp}')
        fp = config.demand_subdir / f"{tp}TTCTransfers.mtx"
        uses_ttc_mat = Matrix.from_emme_mdf(fp)
        uses_ttc[tp] = uses_ttc_mat.tall.sum()
        dly_uses_ttc = dly_uses_ttc + uses_ttc[tp]

with open(transit_dir / "linkedtrips_using_ttc.txt", "w") as f:
    for tp in config.time_periods:
        if config.time_periods[tp]['is_assigned']: 
            f.write(f'{tp}: {uses_ttc[tp]}\n')
    f.write(f'Daily: {dly_uses_ttc}\n')

### Transfers at stations

In [ ]:
from os import PathLike
pd.set_option('future.no_silent_downcasting', True)

def read_transfer_matrix_file(fp: PathLike):
    tm = pd.read_csv(fp, skiprows=[0], index_col=[0])
    tm = tm.drop('total', axis=0)
    tm = tm.drop('total', axis=1)
    tm = tm.replace('-', 0)
    tm = tm.astype(np.float32)
    return tm

def sum_transfers_by_routetype(tm: pd.DataFrame, rgs: dict[str, str], stnname: str) -> pd.DataFrame:
    s = tm.stack()
    s.index.names = ['from_line', 'to_line']
    s.name = 'transfers'
    s = s.reset_index()

    s['stnname'] = stnname
    s['from_grp'] = ''
    s['to_grp'] = ''
    for re_name, re_fltr in rgs.items():
        fltr = s['from_line'].str.contains(re_fltr)
        s.loc[fltr, 'from_grp'] = re_name
        fltr = s['to_line'].str.contains(re_fltr)
        s.loc[fltr, 'to_grp'] = re_name
    return s.groupby(['stnname', 'from_grp', 'to_grp'])['transfers'].sum()

In [ ]:
stn_transfers = {}
dly_stn_transfers = 0
first_tp = True
for tp in config.time_periods:
    if config.time_periods[tp]['is_assigned']: 
        print(f'Processing time period: {tp}')
        transfers_list = []
        stntrans_subdir = config.los_subdir / tp
        for f in stntrans_subdir.glob('transfers_at_*.csv'):
            stnname = f.stem[13:]
            print(f'  Processing station {stnname}')
            tm = read_transfer_matrix_file(f)
            s = sum_transfers_by_routetype(tm, config.stn_transfer_regexprs, stnname)
            if len(s) > 0:
                transfers_list.append(s)
        stn_transfers[tp] = pd.concat(transfers_list)
        if first_tp:
            dly_stn_transfers = stn_transfers[tp].copy()
            first_tp = False
        else:
            dly_stn_transfers = dly_stn_transfers + stn_transfers[tp]
    
for tp, tm in stn_transfers.items():
    tm.to_csv(transit_dir / f"{tp}_stn_transfers.csv")
dly_stn_transfers.to_csv(transit_dir / 'daily_stn_transfers.csv')

# Microsim Results

In [ ]:
ms = MicroSim(config)
ms.read_microsim_files()

In [ ]:
name_col = 'NAME'    # either 'NAME' or 'index_right', depending on the version of pandas used

### Summarize households and persons by region and by PD

In [ ]:
for sa_name, sa_def in [("reg", sa_zns_reg), ("pd", sa_zns_pd)]:
    hhld_totals = ms.summarize_hhld_totals(
        sa_def)
    hhlds_by_income_cat = ms.summarize_hhlds_by_income_cat(
        sa_def)
    hhlds_by_npersons = ms.summarize_hhlds_by_npersons(
        sa_def, combine_above=4)
    hhlds_by_nvehs = ms.summarize_hhlds_by_nvehicles(
        sa_def, combine_above=1)
    pers_totals = ms.summarize_person_totals(
        'por', sa_def)
    pers_agecat = ms.summarize_persons_by_agecat(
        'por', sa_def, age_categories='statcan_5')
    pers_emp_por = ms.summarize_persons_by_empstatus_and_occup(
        'por', sa_def)
    pers_emp_pow = ms.summarize_persons_by_empstatus_and_occup(
        'pow', sa_def)
    final = pd.concat([
        hhld_totals, hhlds_by_income_cat, hhlds_by_npersons, 
        hhlds_by_nvehs, pers_totals, pers_agecat, 
        pers_emp_por, pers_emp_pow
    ], axis=1)
    final.to_csv(ms_dir / f'synthpop_summaries_by_{sa_name}.csv')

### Trip mode shares between regions

In [ ]:
print('Summarizing trips by time period')
dly_trips_by_mode_by_region = None
first_assignment = True
for time_period in ms.time_periods:
    print(f'   Processing time period: {time_period}')
    trips_by_mode_by_region = ms.summarize_trips_by_mode(
        origin_sa=sa_zns_reg, 
        destination_sa=sa_zns_reg, 
        time_period=time_period
    )
    trips_by_region = trips_by_mode_by_region.groupby(
        level=[0, 1], 
        observed=True
    ).sum()

    trips_by_mode_by_region.to_csv(
        ms_dir / f'{time_period}_region_to_region_trips_by_mode.csv')
    trips_by_region.to_csv(
        ms_dir / f'{time_period}_region_to_region_trips.csv')

    if first_assignment:
        dly_trips_by_mode_by_region = trips_by_mode_by_region
        dly_trips_by_region = trips_by_region
        first_assignment = False
    else:
        dly_trips_by_mode_by_region = \
            dly_trips_by_mode_by_region + trips_by_mode_by_region
        dly_trips_by_region = \
            dly_trips_by_region + trips_by_region
dly_trips_by_mode_by_region.to_csv(
    ms_dir / 'daily_region_to_region_trips_by_mode.csv')
dly_trips_by_region.to_csv(
    ms_dir / 'daily_region_to_region_trips.csv')

### Trip mode shares to and from centres

In [ ]:
dly_trips_to_cns = None
first_assignment = True
for time_period in ms.time_periods:
    print(f'Processing time period: {time_period}')
    trips_from_cns = ms.summarize_trips_by_mode(
        origin_sa=sz_zns_cntrs, 
        destination_sa=None, 
        time_period=time_period
    )
    trips_to_cns = ms.summarize_trips_by_mode(
        origin_sa=None, 
        destination_sa=sz_zns_cntrs, 
        time_period=time_period
    )
    trips_to_cns.to_csv(
        ms_dir / f'{time_period}_trips_to_centres_by_mode.csv')
    trips_from_cns.to_csv(
        ms_dir / f'{time_period}_trips_from_centres_by_mode.csv')

    if first_assignment:
        dly_trips_to_cns = trips_to_cns
        dly_trips_from_cns = trips_from_cns
        first_assignment = False
    else:
        dly_trips_to_cns = dly_trips_to_cns + trips_to_cns
        dly_trips_from_cns = dly_trips_from_cns + trips_from_cns
dly_trips_to_cns.to_csv(
    ms_dir / 'daily_trips_to_centres_by_mode.csv')
dly_trips_from_cns.to_csv(
    ms_dir / 'daily_trips_from_centres_by_mode.csv')

# We're done, how long did it all take?

In [ ]:
# Script Runtime in minutes
run_end_time = datetime.now()
run_time = (run_end_time - run_start_time).total_seconds() / 60.0
print("Script run time", run_time) 